In [1]:
import requests
import json
import pandas as pd
from datetime import datetime, date
import time

In [2]:
month_list = pd.date_range(start='2014-01-01', end = '2024-09-23', freq= 'MS' ).strftime("%Y%m%d").tolist()

In [4]:
df = pd.DataFrame()

In [3]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.121 Safari/537.36',
    'Accept': 'application/json, text/plain, */*',
    'Accept-Encoding': 'gzip, deflate, br',
    'Connection': 'keep-alive'
}

In [5]:
with requests.Session() as session:
    session.headers.update(headers)  # 添加 headers
    for month in month_list:
        timestamp = int(time.time() * 1000)#! 後面為模擬隨機產生的參數
        url = f'https://www.twse.com.tw/rwd/zh/TAIEX/MFI94U?date={month}&response=json&_={timestamp}'

        retries = 3
        success = False

        for attempt in range(retries):
            try:
                # 發送請求
                res = session.get(url, timeout=10)  # 使用 session 進行請求
                res.raise_for_status()  # 檢查請求是否成功
                index_json = res.json()

                # 檢查是否有數據
                if 'data' in index_json:
                    index_df = pd.DataFrame.from_dict(index_json['data'])
                    df = pd.concat([df, index_df], ignore_index=True) #! 匯入df
                else:
                    print(f"{month} 沒有資料")

                success = True
                break  # 請求成功後退出重試循環

            except requests.exceptions.ConnectionError as conn_err:
                print(f"連接錯誤（{month}）：{conn_err}。重試中... (第 {attempt + 1} 次)")
                time.sleep(5)  # 增加等待時間至 5 秒後再重試

            except requests.exceptions.HTTPError as http_err:
                print(f"HTTP 錯誤（{month}）：{http_err}")
                break  # 如果是 HTTP 錯誤，則停止重試

            except requests.exceptions.Timeout:
                print(f"{month} 的請求超時。重試中... (第 {attempt + 1} 次)")
                time.sleep(5)

            except Exception as e:
                print(f"其他錯誤（{month}）：{e}")
                break

        if not success:
            print(f"無法獲取 {month} 的數據，跳過該月份。")
        time.sleep(5)  # 請求之間等待更長的時間

# 輸出結果
print(df)

              0          1
0     103/01/02  12,724.76
1     103/01/03  12,627.25
2     103/01/06  12,558.51
3     103/01/07  12,576.67
4     103/01/08  12,641.24
...         ...        ...
2619  113/09/23  49,162.92
2620  113/09/24  49,486.36
2621  113/09/25  50,214.86
2622  113/09/26  50,433.97
2623  113/09/27  50,354.50

[2624 rows x 2 columns]


In [6]:
pip install openpyxl

In [8]:
df.to_csv('index_data.csv', index=False, encoding='utf-8')

In [9]:
df.to_excel('index_data.xlsx', index=False, engine='openpyxl')